# GIZ Internal Request & Ticketing System — Revised Colab Prototype

**Social Transformation Cluster**

This notebook translates the revised concept note and Janosch's feedback into a practical prototype.

### Revised first-pilot scope
- Internal **procurement requests**
- Internal **contract requests**
- **Procurement/Contract clarification** requests
- Limited **general procurement/contract support**

Vendor-facing/external requests are intentionally outside the first prototype.

### Important
This notebook is a **design and demonstration prototype only**. It does not connect to GIZ systems and must not be used with confidential GIZ information.
The intended production direction is the approved GIZ Microsoft 365 environment: **MS Forms / MS Lists → Power Automate → Procurement/Contract processing → Power BI**.
KIM is represented by a transparent mock assistant until the actual KIM/API options are confirmed by the responsible IT team.


## 1. Setup

In [ ]:
# Colab / Jupyter setup
!pip -q install pandas numpy plotly openpyxl

import os
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

random.seed(42)
np.random.seed(42)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Prototype environment is ready.")


## 2. Revised scope and controlled definitions

The first prototype deliberately starts narrow. Category, request type and priority are separate fields:

- **Category** = responsible service area.
- **Sub-category / request type** = specific work requested.
- **Priority** = business impact + deadline pressure + dependency.
- **Status** = where the ticket is in the lifecycle.


In [ ]:
CATEGORIES = {
    "Procurement Request": {
        "queue": "Procurement",
        "definition": "A request to initiate, support or progress a procurement activity.",
        "subcategories": [
            "New procurement",
            "Quotation / RFQ support",
            "Tender / sourcing support",
            "Procurement document review",
            "Procurement planning support",
            "Other procurement action",
        ],
    },
    "Contract Request": {
        "queue": "Contracts",
        "definition": "A request requiring action on a contract or agreement.",
        "subcategories": [
            "New contract review",
            "Contract amendment",
            "Contract extension",
            "Contract interpretation / review",
            "Contract closure-related action",
            "Other contract action",
        ],
    },
    "Procurement/Contract Clarification": {
        "queue": "Coordinator Triage",
        "definition": "A question seeking guidance where no substantive procurement/contract action is yet requested.",
        "subcategories": [
            "Procedure clarification",
            "Required documents clarification",
            "Previous instruction clarification",
            "Status / process clarification",
        ],
    },
    "General Procurement/Contract Support": {
        "queue": "Coordinator Triage",
        "definition": "Internal support directly connected to procurement/contract operations but not covered above.",
        "subcategories": [
            "Record / information support",
            "Process support",
            "Reporting support",
            "Other support",
        ],
    },
}

PRIORITIES = ["Normal", "High", "Urgent"]

STATUS_FLOW = [
    "NEW",
    "RECEIVED",
    "UNDER REVIEW",
    "CLARIFICATION REQUIRED",
    "CLARIFICATION RECEIVED",
    "ASSIGNED",
    "IN PROCESS",
    "PENDING APPROVAL",
    "RESOLVED",
    "CLOSED",
]

EXCEPTION_STATUSES = ["ON HOLD", "REJECTED", "CANCELLED", "ESCALATED"]

print(f"Categories in pilot: {len(CATEGORIES)}")
print("Statuses:", " → ".join(STATUS_FLOW))


## 3. Category classification and sub-category suggestion

The classifier below is deliberately transparent. It is **not an AI model**. It is a stand-in for future KIM functionality and provides an auditable baseline for testing the workflow.


In [ ]:
CATEGORY_KEYWORDS = {
    "Procurement Request": [
        "procure", "procurement", "purchase", "buy", "quotation", "rfq",
        "tender", "sourcing", "goods", "supplies", "equipment", "specification"
    ],
    "Contract Request": [
        "contract", "agreement", "amendment", "extension", "renewal",
        "contract review", "contract reference"
    ],
    "Procurement/Contract Clarification": [
        "clarify", "clarification", "what documents", "which process",
        "not sure", "explain", "procedure", "requirement"
    ],
}

SUBCATEGORY_KEYWORDS = {
    "Procurement Request": {
        "Quotation / RFQ support": ["quotation", "rfq", "request for quotation"],
        "Tender / sourcing support": ["tender", "sourcing", "bid"],
        "New procurement": ["new procurement", "purchase", "procure", "buy"],
        "Procurement document review": ["specification", "procurement document", "review"],
    },
    "Contract Request": {
        "Contract amendment": ["amendment", "change contract"],
        "Contract extension": ["extension", "extend", "renewal"],
        "New contract review": ["new contract", "draft contract", "service contract"],
        "Contract interpretation / review": ["contract review", "interpretation"],
    },
    "Procurement/Contract Clarification": {
        "Procedure clarification": ["which process", "procedure", "how do i"],
        "Required documents clarification": ["what documents", "required documents"],
        "Previous instruction clarification": ["previous instruction", "last advice"],
    },
}

def classify_request(text):
    text = str(text).lower()

    # Specific categories first
    scores = {category: 0 for category in CATEGORIES}
    for category, keywords in CATEGORY_KEYWORDS.items():
        scores[category] = sum(1 for word in keywords if word in text)

    best = max(scores, key=scores.get)
    if scores[best] == 0:
        best = "General Procurement/Contract Support"

    subcategory = "Other support"
    for sub, keywords in SUBCATEGORY_KEYWORDS.get(best, {}).items():
        if any(word in text for word in keywords):
            subcategory = sub
            break

    return best, subcategory, scores[best]

tests = [
    "We need to procure laptops for the project.",
    "Please review the amendment to the service contract.",
    "Which documents are required for this procurement?",
    "I need support with the procurement record.",
]

for text in tests:
    print(text)
    print(" ->", classify_request(text))


## 4. Routing logic

The pilot uses **semi-automated routing**:

1. The system suggests a category and queue.
2. A ticketing coordinator reviews the suggestion.
3. The coordinator confirms or corrects the routing.
4. The named processor is assigned using category, specialization and workload.

This avoids assuming that fully automatic routing is reliable before the pilot has generated evidence.


In [ ]:
PROCESSORS = pd.DataFrame([
    {"processor": "Processor A", "queue": "Procurement", "specialization": "Goods", "capacity": 10},
    {"processor": "Processor B", "queue": "Procurement", "specialization": "Services", "capacity": 10},
    {"processor": "Processor C", "queue": "Procurement", "specialization": "Tender", "capacity": 8},
    {"processor": "Processor D", "queue": "Contracts", "specialization": "Contracts", "capacity": 10},
    {"processor": "Processor E", "queue": "Contracts", "specialization": "Agreements", "capacity": 8},
])

def suggest_route(category, subcategory=None, project=None):
    queue = CATEGORIES[category]["queue"]

    if queue in ["Coordinator Triage"]:
        return {
            "queue": queue,
            "processor": None,
            "routing_mode": "Coordinator review required",
            "reason": "Category does not identify a single processing owner."
        }

    candidates = PROCESSORS[PROCESSORS["queue"] == queue].copy()
    processor = candidates.sort_values(["capacity", "processor"], ascending=[False, True]).iloc[0]["processor"]

    return {
        "queue": queue,
        "processor": processor,
        "routing_mode": "Rule-based suggestion + human confirmation",
        "reason": f"Category={category}; queue={queue}"
    }

for category in CATEGORIES:
    print(category, "->", suggest_route(category))


## 5. Priority model

Priority is not simply whatever the requester selects. The prototype asks for three factors:

- **Impact**: how strongly the request affects operations/project delivery.
- **Deadline pressure**: how close the required date is.
- **Dependency**: whether other activities are blocked.

The score provides a **suggestion**. The receiver can override it and record the reason.


In [ ]:
def priority_score(impact, deadline_pressure, dependency):
    score = int(impact) + int(deadline_pressure) + int(dependency)

    if score >= 7:
        suggested = "Urgent"
    elif score >= 5:
        suggested = "High"
    else:
        suggested = "Normal"

    return score, suggested

examples = [
    {"impact": 1, "deadline_pressure": 1, "dependency": 0},
    {"impact": 2, "deadline_pressure": 2, "dependency": 1},
    {"impact": 3, "deadline_pressure": 3, "dependency": 2},
]

for e in examples:
    print(e, "->", priority_score(**e))


## 6. Ticket data model

The production concept maps naturally to an MS Lists Tickets table plus a separate Ticket History table.


In [ ]:
TICKET_FIELDS = [
    "ticket_id", "request_datetime", "title", "description",
    "requester_name", "department_project", "contact",
    "category", "sub_category", "request_type",
    "priority", "priority_score", "priority_reason",
    "service_queue", "assigned_person", "routing_mode",
    "status",
    "first_response_date", "clarification_requested", "clarification_date",
    "approval_required", "approval_date", "processing_start_date",
    "resolution_date", "closure_date",
    "response_time_hrs", "resolution_time_hrs",
    "clarification_cycles", "escalated", "sla_status",
    "resolution_category", "resolution_description", "root_cause", "closure_reason"
]

HISTORY_FIELDS = [
    "history_id", "ticket_id", "timestamp", "actor",
    "action", "field_name", "old_value", "new_value", "comment"
]

print("Ticket fields:", len(TICKET_FIELDS))
print("History fields:", len(HISTORY_FIELDS))


## 7. Ticket creation

The function creates a ticket with a unique ID and records the first history event.


In [ ]:
ticket_counter = 1
history_counter = 1
history = []

def generate_ticket_id(dt, sequence):
    return f"TCK-{dt.strftime('%Y%m%d')}-{sequence:04d}"

def add_history(ticket_id, actor, action, field_name=None, old_value=None, new_value=None, comment=""):
    global history_counter
    history.append({
        "history_id": history_counter,
        "ticket_id": ticket_id,
        "timestamp": datetime.now(),
        "actor": actor,
        "action": action,
        "field_name": field_name,
        "old_value": old_value,
        "new_value": new_value,
        "comment": comment,
    })
    history_counter += 1

def create_ticket(
    requester_name,
    department_project,
    contact,
    title,
    description,
    impact=1,
    deadline_pressure=1,
    dependency=0,
    requester_priority=None,
    request_datetime=None,
):
    global ticket_counter

    request_datetime = request_datetime or datetime.now()
    category, subcategory, confidence = classify_request(description)
    route = suggest_route(category, subcategory, department_project)

    score, suggested_priority = priority_score(
        impact, deadline_pressure, dependency
    )

    # Human review is required when requester priority differs from the system suggestion.
    final_priority = requester_priority or suggested_priority
    priority_reason = (
        f"Impact={impact}, deadline={deadline_pressure}, dependency={dependency}; "
        f"system suggestion={suggested_priority}"
    )

    ticket_id = generate_ticket_id(request_datetime, ticket_counter)

    ticket = {
        "ticket_id": ticket_id,
        "request_datetime": request_datetime,
        "title": title,
        "description": description,
        "requester_name": requester_name,
        "department_project": department_project,
        "contact": contact,
        "category": category,
        "sub_category": subcategory,
        "request_type": subcategory,
        "priority": final_priority,
        "priority_score": score,
        "priority_reason": priority_reason,
        "service_queue": route["queue"],
        "assigned_person": route["processor"],
        "routing_mode": route["routing_mode"],
        "status": "NEW",
        "first_response_date": None,
        "clarification_requested": False,
        "clarification_date": None,
        "approval_required": category in ["Procurement Request", "Contract Request"],
        "approval_date": None,
        "processing_start_date": None,
        "resolution_date": None,
        "closure_date": None,
        "response_time_hrs": None,
        "resolution_time_hrs": None,
        "clarification_cycles": 0,
        "escalated": False,
        "sla_status": None,
        "resolution_category": None,
        "resolution_description": None,
        "root_cause": None,
        "closure_reason": None,
    }

    add_history(ticket_id, "System", "TICKET_CREATED", "status", None, "NEW")
    ticket_counter += 1
    return ticket

example_ticket = create_ticket(
    requester_name="Example Requester",
    department_project="Example Project",
    contact="example@example.org",
    title="Procure laptops",
    description="We need to procure laptops for the project.",
    impact=2,
    deadline_pressure=2,
    dependency=1,
)

pd.DataFrame([example_ticket])


## 8. Lifecycle automation

The lifecycle is recorded as events rather than requiring the processor to manually maintain every performance field.


In [ ]:
VALID_NEXT = {
    "NEW": ["RECEIVED", "UNDER REVIEW", "CANCELLED"],
    "RECEIVED": ["UNDER REVIEW", "CANCELLED"],
    "UNDER REVIEW": ["CLARIFICATION REQUIRED", "ASSIGNED", "REJECTED", "CANCELLED"],
    "CLARIFICATION REQUIRED": ["CLARIFICATION RECEIVED", "CANCELLED"],
    "CLARIFICATION RECEIVED": ["UNDER REVIEW", "ASSIGNED"],
    "ASSIGNED": ["IN PROCESS", "CLARIFICATION REQUIRED", "ON HOLD", "ESCALATED"],
    "IN PROCESS": ["PENDING APPROVAL", "RESOLVED", "CLARIFICATION REQUIRED", "ON HOLD", "ESCALATED"],
    "PENDING APPROVAL": ["IN PROCESS", "RESOLVED", "ESCALATED"],
    "RESOLVED": ["CLOSED", "IN PROCESS"],
    "CLOSED": [],
    "ON HOLD": ["IN PROCESS", "CANCELLED"],
    "REJECTED": [],
    "CANCELLED": [],
    "ESCALATED": ["ASSIGNED", "IN PROCESS", "RESOLVED"],
}

def change_status(ticket, new_status, actor="Processor", comment=""):
    old_status = ticket["status"]
    allowed = VALID_NEXT.get(old_status, [])

    if new_status not in allowed:
        raise ValueError(f"Invalid transition: {old_status} -> {new_status}")

    now = datetime.now()
    ticket["status"] = new_status

    if new_status == "RECEIVED" and ticket["first_response_date"] is None:
        ticket["first_response_date"] = now
        ticket["response_time_hrs"] = round(
            (now - pd.Timestamp(ticket["request_datetime"])).total_seconds() / 3600, 2
        )

    if new_status == "CLARIFICATION REQUIRED":
        ticket["clarification_requested"] = True
        ticket["clarification_date"] = now
        ticket["clarification_cycles"] += 1

    if new_status == "IN PROCESS" and ticket["processing_start_date"] is None:
        ticket["processing_start_date"] = now

    if new_status == "RESOLVED":
        ticket["resolution_date"] = now
        ticket["resolution_time_hrs"] = round(
            (now - pd.Timestamp(ticket["request_datetime"])).total_seconds() / 3600, 2
        )

    if new_status == "CLOSED":
        ticket["closure_date"] = now
        ticket["closure_reason"] = ticket["closure_reason"] or "Request completed"

    add_history(
        ticket["ticket_id"], actor, "STATUS_CHANGED",
        "status", old_status, new_status, comment
    )

# Demonstrate a valid lifecycle
change_status(example_ticket, "RECEIVED", actor="Coordinator")
change_status(example_ticket, "UNDER REVIEW", actor="Coordinator")
change_status(example_ticket, "ASSIGNED", actor="Coordinator")
change_status(example_ticket, "IN PROCESS", actor="Processor")
change_status(example_ticket, "RESOLVED", actor="Processor", comment="Action completed")
change_status(example_ticket, "CLOSED", actor="Coordinator")

print(example_ticket["ticket_id"], example_ticket["status"])


## 9. SLA and performance logic

Illustrative prototype targets only. Final targets must be agreed before production.


In [ ]:
SLA_TARGET_HOURS = {
    "Urgent": 24,
    "High": 72,
    "Normal": 168,
}

def calculate_sla(ticket):
    if ticket["resolution_time_hrs"] is None:
        return "Open"

    target = SLA_TARGET_HOURS[ticket["priority"]]
    return "Met" if ticket["resolution_time_hrs"] <= target else "Breached"

ticket_df_one = pd.DataFrame([example_ticket])
ticket_df_one["sla_status"] = ticket_df_one.apply(calculate_sla, axis=1)
display(ticket_df_one[[
    "ticket_id", "category", "sub_category", "priority",
    "status", "response_time_hrs", "resolution_time_hrs", "sla_status"
]])


## 10. Import the supplied sample ticket dataset

If `tickets_dataset_sample.csv` is uploaded to the Colab session, the notebook will analyze it. Otherwise, it creates a synthetic dataset for demonstration.

**The supplied sample is treated as prototype data; it is not assumed to represent actual organizational performance.**


In [ ]:
SAMPLE_FILE = "tickets_dataset_sample.csv"

def load_sample_or_generate():
    if os.path.exists(SAMPLE_FILE):
        sample = pd.read_csv(SAMPLE_FILE)
        print(f"Loaded supplied sample dataset: {len(sample):,} rows")
        return sample

    print("Sample CSV not found. Generating synthetic prototype data.")
    return pd.DataFrame()

df = load_sample_or_generate()
display(df.head())
print("Columns:", len(df.columns))


## 11. Normalize the supplied data for the revised model

The original sample can contain broader categories. The following mapping keeps the analysis aligned with the revised pilot scope.


In [ ]:
CATEGORY_MAP = {
    "Procurement-related request": "Procurement Request",
    "Contract-related request": "Contract Request",
    "Clarification request": "Procurement/Contract Clarification",
    "Other service-related request": "General Procurement/Contract Support",
    "Vendor-related request": "OUT_OF_SCOPE_VENDOR",
    "System or administrative issue": "OUT_OF_SCOPE_SYSTEM",
}

if not df.empty and "category" in df.columns:
    df["revised_category"] = df["category"].map(CATEGORY_MAP).fillna("General Procurement/Contract Support")
    df["pilot_scope"] = np.where(
        df["revised_category"].str.startswith("OUT_OF_SCOPE"),
        "Outside first prototype",
        "In pilot scope"
    )

    display(
        df.groupby(["revised_category", "pilot_scope"])
          .size()
          .reset_index(name="tickets")
          .sort_values("tickets", ascending=False)
    )
else:
    print("No dataset available for normalization.")


## 12. Prototype ticket generator for dashboard testing

When real operational data is not available, synthetic data can be used to test the dashboard. These values are explicitly illustrative.


In [ ]:
def generate_synthetic_tickets(n=260):
    rng = np.random.default_rng(42)
    categories = list(CATEGORIES.keys())
    projects = [
        "Project A", "Project B", "Project C", "Social Transformation Cluster",
        "Health Programme", "Digital Project"
    ]
    processors = ["Processor A", "Processor B", "Processor C", "Processor D", "Processor E"]

    rows = []
    start = datetime.now() - timedelta(days=180)

    for i in range(1, n + 1):
        category = rng.choice(categories, p=[0.40, 0.30, 0.15, 0.15])
        sub = rng.choice(CATEGORIES[category]["subcategories"])
        priority = rng.choice(PRIORITIES, p=[0.55, 0.30, 0.15])
        status = rng.choice(
            STATUS_FLOW,
            p=[0.03, 0.04, 0.08, 0.05, 0.03, 0.07, 0.25, 0.08, 0.10, 0.27]
        )

        req_dt = start + timedelta(
            days=int(rng.integers(0, 180)),
            hours=int(rng.integers(0, 24)),
            minutes=int(rng.integers(0, 60))
        )

        response = round(float(max(0.5, rng.gamma(2, 5))), 1)
        resolution = round(float(response + max(2, rng.gamma(3, 10))), 1)
        target = SLA_TARGET_HOURS[priority]

        rows.append({
            "ticket_id": f"TCK-SYN-{i:04d}",
            "request_datetime": req_dt,
            "title": f"Prototype {category} request",
            "description": f"Synthetic example for {sub}",
            "requester_name": f"Requester {int(rng.integers(1, 41))}",
            "department_project": rng.choice(projects),
            "category": category,
            "sub_category": sub,
            "priority": priority,
            "service_queue": CATEGORIES[category]["queue"],
            "assigned_person": rng.choice(processors),
            "status": status,
            "response_time_hrs": response,
            "resolution_time_hrs": resolution if status in ["RESOLVED", "CLOSED"] else np.nan,
            "clarification_cycles": int(rng.choice([0, 1, 2], p=[0.70, 0.25, 0.05])),
            "escalated": bool(rng.random() < 0.08),
            "sla_status": "Met" if resolution <= target else "Breached",
        })

    return pd.DataFrame(rows)

dashboard_df = generate_synthetic_tickets(260)
print(f"Synthetic dashboard records: {len(dashboard_df):,}")
display(dashboard_df.head())


## 13. Executive dashboard

In [ ]:
closed = dashboard_df["status"].isin(["RESOLVED", "CLOSED"])
open_mask = ~closed

kpis = {
    "Total tickets": len(dashboard_df),
    "Open tickets": int(open_mask.sum()),
    "Closed tickets": int((dashboard_df["status"] == "CLOSED").sum()),
    "Escalated tickets": int(dashboard_df["escalated"].sum()),
    "Average response (hrs)": round(dashboard_df["response_time_hrs"].mean(), 1),
    "Average resolution (hrs)": round(dashboard_df["resolution_time_hrs"].mean(), 1),
    "SLA compliance (%)": round((dashboard_df["sla_status"] == "Met").mean() * 100, 1),
    "Avg clarification cycles": round(dashboard_df["clarification_cycles"].mean(), 2),
}

display(pd.DataFrame({"KPI": list(kpis), "Value": list(kpis.values())}))

fig = go.Figure()
fig.add_trace(go.Indicator(
    mode="number",
    value=kpis["Total tickets"],
    title={"text": "Total Tickets"},
    domain={"row": 0, "column": 0}
))
fig.add_trace(go.Indicator(
    mode="number",
    value=kpis["Open tickets"],
    title={"text": "Open Tickets"},
    domain={"row": 0, "column": 1}
))
fig.add_trace(go.Indicator(
    mode="number",
    value=kpis["Closed tickets"],
    title={"text": "Closed Tickets"},
    domain={"row": 0, "column": 2}
))
fig.add_trace(go.Indicator(
    mode="number",
    value=kpis["SLA compliance (%)"],
    number={"suffix": "%"},
    title={"text": "SLA Compliance"},
    domain={"row": 0, "column": 3}
))
fig.update_layout(
    grid={"rows": 1, "columns": 4},
    title="Internal Ticketing — Executive Overview",
    height=280
)
fig.show()


## 14. Workload dashboard

In [ ]:
fig1 = px.bar(
    dashboard_df["category"].value_counts().reset_index(),
    x="category", y="count",
    title="Tickets by Category",
    labels={"count": "Tickets", "category": "Category"}
)
fig1.update_xaxes(tickangle=25)
fig1.show()

fig2 = px.bar(
    dashboard_df["service_queue"].value_counts().reset_index(),
    x="service_queue", y="count",
    title="Tickets by Service Queue",
    labels={"count": "Tickets", "service_queue": "Service Queue"}
)
fig2.show()

fig3 = px.bar(
    dashboard_df["priority"].value_counts().reindex(PRIORITIES).fillna(0).reset_index(),
    x="priority", y="count",
    title="Tickets by Priority",
    labels={"count": "Tickets", "priority": "Priority"}
)
fig3.show()


## 15. Performance and backlog dashboard

In [ ]:
performance = (
    dashboard_df.groupby("category")
    .agg(
        tickets=("ticket_id", "count"),
        avg_response_hours=("response_time_hrs", "mean"),
        avg_resolution_hours=("resolution_time_hrs", "mean"),
        avg_clarification_cycles=("clarification_cycles", "mean"),
        escalated=("escalated", "sum"),
    )
    .reset_index()
)

display(performance.round(2))

fig = px.bar(
    performance.sort_values("avg_resolution_hours", ascending=False),
    x="category", y="avg_resolution_hours",
    title="Average Resolution Time by Category",
    labels={"avg_resolution_hours": "Hours", "category": "Category"}
)
fig.update_xaxes(tickangle=25)
fig.show()

aging = dashboard_df.loc[open_mask].copy()
aging["age_days"] = (
    pd.Timestamp.now() - pd.to_datetime(aging["request_datetime"])
).dt.total_seconds() / 86400

aging_bands = pd.cut(
    aging["age_days"],
    bins=[-1, 2, 7, 14, float("inf")],
    labels=["0–2 days", "3–7 days", "8–14 days", ">14 days"]
).value_counts().sort_index()

fig = px.bar(
    aging_bands.reset_index(),
    x="age_days", y="count",
    title="Open Ticket Aging",
    labels={"age_days": "Age", "count": "Open Tickets"}
)
fig.show()


## 16. Process-improvement analysis

The data should help identify where time is being consumed and where clarification is recurring.


In [ ]:
process_improvement = (
    dashboard_df.groupby(["category", "sub_category"])
    .agg(
        tickets=("ticket_id", "count"),
        avg_resolution_hours=("resolution_time_hrs", "mean"),
        avg_clarification_cycles=("clarification_cycles", "mean"),
        escalated=("escalated", "sum")
    )
    .reset_index()
    .sort_values(["avg_clarification_cycles", "avg_resolution_hours"], ascending=False)
)

display(process_improvement.round(2).head(15))


## 17. Mock KIM-guided intake

This simulates the concept proposed in the revised note:

**Requester → KIM assistant → targeted questions → completeness check → category/sub-category suggestion → priority support → ticket preparation**

It is intentionally rule-based and transparent. It is not an actual KIM connection.


In [ ]:
KIM_REQUIRED_FIELDS = {
    "Procurement Request": [
        "project_or_cost_centre",
        "estimated_value",
        "required_date",
        "budget_available",
        "technical_specification"
    ],
    "Contract Request": [
        "project_or_cost_centre",
        "contract_reference",
        "required_action",
        "required_date"
    ],
    "Procurement/Contract Clarification": [
        "question",
        "related_ticket_id"
    ],
    "General Procurement/Contract Support": [
        "project_or_cost_centre",
        "support_needed"
    ],
}

def kim_intake(description, answers):
    category, subcategory, confidence = classify_request(description)
    required = KIM_REQUIRED_FIELDS[category]
    missing = [field for field in required if not answers.get(field)]

    return {
        "suggested_category": category,
        "suggested_subcategory": subcategory,
        "classification_confidence": confidence,
        "missing_fields": missing,
        "complete": len(missing) == 0,
        "next_step": "Ready for ticket submission" if not missing else "Ask targeted follow-up questions",
    }

demo_answers = {
    "project_or_cost_centre": "Project A",
    "estimated_value": "EUR 5000",
    "required_date": "2026-09-15",
    "budget_available": "Yes",
    "technical_specification": "Laptop specification attached",
}

kim_result = kim_intake(
    "We need to procure laptops for the project.",
    demo_answers
)

display(pd.DataFrame([kim_result]))


## 18. Simulated Ticket History / Audit Trail

In [ ]:
history_df = pd.DataFrame(history)

if not history_df.empty:
    display(history_df.sort_values("timestamp"))
else:
    print("No history records yet.")


## 19. Proposed MS Lists / Power Automate mapping

This is the bridge from the Colab prototype to the intended GIZ-approved technical environment.


In [ ]:
architecture = pd.DataFrame([
    ["MS Forms / List Form", "Request intake", "Requester fields, description, attachments, urgency inputs"],
    ["MS Lists – Tickets", "Current ticket record", "Ticket ID, category, owner, status, priority, dates, resolution"],
    ["MS Lists – Ticket History", "Audit trail", "Ticket ID, timestamp, actor, action, old/new value"],
    ["MS Lists – Routing Matrix", "Controlled routing", "Category, sub-category, queue, processor, active flag"],
    ["Power Automate", "Automation", "Create ID, notify, route, timestamp, remind, escalate"],
    ["Power BI", "Reporting", "KPI, workload, backlog, performance, process improvement"],
    ["KIM", "Optional assisted intake", "Understand, ask, classify, check completeness"],
], columns=["Component", "Purpose", "Prototype mapping"])

display(architecture)


## 20. Export prototype outputs

In [ ]:
export_df = dashboard_df.copy()

export_df.to_csv("giz_ticketing_prototype_dashboard_data.csv", index=False)

if not history_df.empty:
    history_df.to_csv("giz_ticketing_prototype_history.csv", index=False)

pd.DataFrame([kpis]).to_csv("giz_ticketing_kpi_summary.csv", index=False)

print("Prototype exports created:")
print("- giz_ticketing_prototype_dashboard_data.csv")
print("- giz_ticketing_kpi_summary.csv")
if not history_df.empty:
    print("- giz_ticketing_prototype_history.csv")


## 21. Recommended implementation sequence

**Standardize → Digitize → Measure → Analyze → Improve → Automate with KIM-supported AI**

### Immediate next step
Validate the following with the Procurement/Contract Service Unit:
1. Four pilot categories and their sub-categories.
2. Routing matrix and named coordinator/receivers.
3. Priority definitions and final SLA targets.
4. Mandatory request fields.
5. Status transition rules.
6. Which Power Automate actions can be automated.
7. KIM/API feasibility with the responsible IT team.

The Colab prototype should then be used as a reference for configuring the Microsoft 365 pilot.
